# M2 · Collect expert traces

**Goal:** record which experts OLMoE uses for ~200k tokens of **English (WikiText)**
and ~200k tokens of **Python code**. This is the only heavy GPU step;
everything after it (M4–M7) runs on the CPU using these files.

**Kaggle:** GPU T4 x1, Internet On. About 30–60 minutes.

### Trace file format
`traces_<dataset>.npz` holds one array `trace` of shape **[tokens, 16 layers, 8 experts]**.
`trace[t, l]` = the 8 experts that token `t` used in layer `l`.

### Why one forward pass is enough
The model is *causal*: token t only looks at tokens before it. So running the
whole 512-token chunk at once picks exactly the same experts as generating
the text one token at a time.

In [ ]:
# ---- Setup: get the latest project code (run this cell first, every session) ----
import os, sys

REPO_URL = "https://github.com/YOUR-TEAM/moe-offload.git"   # <-- change to your team's repo
# Private repo? Add a Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets), then uncomment:
# from kaggle_secrets import UserSecretsClient
# REPO_URL = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")

if os.path.exists("/kaggle"):                      # running on Kaggle
    import glob, shutil
    CODE_DIR = "/kaggle/working/moe-offload"
    # Option A: the code was uploaded as a Kaggle dataset ("moe-offload-code") and added as an input
    uploaded = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/moe-offload-code/config.py", recursive=True)]
    if uploaded:
        shutil.copytree(uploaded[0], CODE_DIR, dirs_exist_ok=True)
        print("using code from dataset:", uploaded[0])
    # Option B: get the code from GitHub
    elif not os.path.exists(CODE_DIR):
        !git clone -q {REPO_URL} {CODE_DIR}         # first time: download the code
    else:
        !git -C {CODE_DIR} pull -q                  # later: get the newest commits
    os.chdir(CODE_DIR)
    !pip install -q -r requirements.txt
else:                                              # running on your own computer
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")                             # go to the repo root

sys.path.insert(0, os.getcwd())                    # so "import config" and "from src..." work
!git log -1 --format="Code version: %h  %s"

In [ ]:
import os
import config
from src.traces import load_moe, collect_traces
from src.data import make_chunks

model, tok = load_moe()

## Collect
`collect_traces` saves progress every 10 batches. If Kaggle disconnects,
just run the notebook again: it continues where it stopped.

In [ ]:
for dataset in ["wikitext", "code"]:
    chunks = make_chunks(tok, dataset)            # [n_chunks, 512] token ids
    print(f"\n{dataset}: {chunks.shape[0]} chunks x {chunks.shape[1]} tokens")
    out_path = os.path.join(config.TRACE_DIR, f"traces_{dataset}.npz")
    trace = collect_traces(model, chunks, out_path)
    print("saved", out_path, "shape", trace.shape)

## Quick look at the result

In [ ]:
import numpy as np
from src.files import load_trace

for dataset in ["wikitext", "code"]:
    tr = load_trace(dataset)
    print(dataset, "tokens:", tr.shape[0], "| first token, layer 0 experts:", tr[0, 0].tolist())

## Save the traces as a Kaggle Dataset (do this once)
1. Click **Save Version → Save & Run All** (so the files are kept as notebook output).
2. Open the notebook's **Output** tab → **New Dataset** → name it **`moe-traces`**.
3. In M4–M8: **Add Input → Datasets → moe-traces**. `find_file()` will find the files.

## ✅ Done when
`traces_wikitext.npz` and `traces_code.npz` are saved in the `moe-traces` dataset.